# Music Annotation Project — Notebook 31
## V8.2b Interactive Gradio Six-Band Confidence Explorer with Click Inspector

**Runtime:** CPU is sufficient.

This is a development-only diagnostic. It reads the completed V8.2 source-group OOF probabilities and does **not** retrain, refit, recalibrate, or change corrected pages.

### Added click inspector
Click any of the three synchronised page views to inspect the corresponding page coordinate. The inspector reports:

- clicked display coordinate and mapped full-page `(x, y)`;
- exact eligible-candidate coordinate (or an explicitly labelled nearest eligible candidate when display dilation was clicked);
- calibrated `p_print`, raw score, fold, source group, saved decision, truth label, and current six-band assignment;
- the exact 33×33 observed patch centred on the candidate;
- the complete 11-channel 33×33 tensor used by V8.2, reconstructed with the frozen preprocessing and reflect padding;
- per-channel centre values, minima, maxima, and means.

Truth is shown only as a post-hoc development diagnostic. It is never used to generate probabilities or page decisions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=5.0" opencv-python-headless pandas numpy pyarrow pillow matplotlib

## 1. Configuration and source resolution

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import gradio as gr

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
EXPLORER_ROOT = ROOT / 'observed_print_classifier_v8_2b_gradio_explorer'
SOURCE_V82_RUN_OVERRIDE = None

PATCH = 33
HALF = PATCH // 2
CHANNELS = ['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']

# Six diagnostic display states need five breakpoints.
DEFAULT_BREAKPOINTS_PERCENT = [10, 35, 50, 65, 90]
DEFAULT_ALPHA = 0.62
DEFAULT_RADIUS = 1
DEFAULT_ZOOM = 1.0
DEFAULT_PAN_X = 50
DEFAULT_PAN_Y = 50
VIEWPORT_MAX_WIDTH = 1300
VIEWPORT_MAX_HEIGHT = 850

# A coloured display radius can place colour on a neighbouring non-eligible pixel.
# If that happens, the inspector may identify the closest eligible source pixel,
# but it reports both coordinates and never silently treats them as identical.
SNAP_TO_NEAREST_ELIGIBLE = True
MAX_SNAP_DISTANCE = 5.0

STATE_NAMES = [
    'Definitely annotation',
    'Probably annotation',
    'Weak annotation lean',
    'Weak print lean',
    'Probably print',
    'Definitely print',
]
# RGB colours for Gradio/PIL.
STATE_RGB = [
    (205, 0, 0),
    (255, 128, 0),
    (35, 100, 220),
    (0, 200, 220),
    (95, 255, 45),
    (0, 155, 0),
]
TRUTH_NAMES = {0:'ANNOTATION', 1:'PRINT', 2:'UNCERTAIN'}

REQUIRED = [
    'experiment_identity.json',
    'tables/all_eligible_oof_predictions.parquet',
    'inputs/channel_schema.json',
]

def complete_runs(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED):
                hits.append(p)
    return sorted(hits, key=lambda p:p.stat().st_mtime)

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82_RUN_OVERRIDE)
    missing=[r for r in REQUIRED if not (SOURCE/r).exists()]
    if missing:
        raise RuntimeError(f'Incomplete V8.2 source override: {missing}')
else:
    runs=complete_runs(V82_ROOT)
    if not runs:
        raise RuntimeError(f'No complete V8.2 run found below {V82_ROOT}')
    SOURCE=runs[-1]

IDENTITY=json.loads((SOURCE/'experiment_identity.json').read_text())
if IDENTITY.get('version') != '8.2-development-whole-page':
    raise RuntimeError(f'Unexpected V8.2 source identity: {IDENTITY.get("version")}')
if IDENTITY.get('independent_data_used') is not False:
    raise RuntimeError('The explorer may use only the development-only V8.2 OOF run.')

FULL=pd.read_parquet(SOURCE/'tables'/'all_eligible_oof_predictions.parquet')
required_columns={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required_columns-set(FULL.columns)
if missing:
    raise RuntimeError(f'OOF prediction table missing columns: {sorted(missing)}')
for c in ['y','x','truth','fold']:
    FULL[c]=pd.to_numeric(FULL[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:
    FULL[c]=pd.to_numeric(FULL[c],errors='raise').astype(np.float64)
if not np.isfinite(FULL[['raw_print','p_print']].to_numpy()).all():
    raise RuntimeError('Non-finite V8.2 scores found.')
if FULL.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x rows found.')
FULL['page']=FULL.page.astype(str)
PAGES=sorted(FULL.page.unique().tolist())
PAGE_ROWS={p:g.copy().reset_index(drop=True) for p,g in FULL.groupby('page',sort=False)}
PAGE_LOOKUP={p:{(int(r.x),int(r.y)):i for i,r in g.iterrows()} for p,g in PAGE_ROWS.items()}

V75=Path(IDENTITY['v7_5_reference'])
status=pd.read_csv(V75/'tables'/'cache_status.csv')
CACHE={str(r.page):Path(r.cache) for _,r in status.iterrows()}
for page in PAGES:
    if page not in CACHE:
        raise RuntimeError(f'No V7.5 cache found for {page}')
    for name in ['observed.png','v7_8.png','v8_2.png']:
        if not (SOURCE/'whole_pages'/page/name).exists():
            raise RuntimeError(f'{page}: missing {name}')

EXPLORER_ROOT.mkdir(parents=True,exist_ok=True)
SESSION_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_click_inspector'
SESSION_DIR=EXPLORER_ROOT/SESSION_ID
SESSION_DIR.mkdir(parents=True,exist_ok=False)
print('Source V8.2:', SOURCE)
print('Pages:',len(PAGES),'OOF rows:',len(FULL))
print('Session output:',SESSION_DIR)

## 2. Exact V8.2 page channels and 33×33 tensor reconstruction

In [ ]:
PAGE_RUNTIME_CACHE={}

def background_surface(obs,mask,sigma=45.0):
    vis=(~mask)&(obs>=200)
    num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma)
    den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def load_page_runtime(page):
    if page in PAGE_RUNTIME_CACHE:
        return PAGE_RUNTIME_CACHE[page]
    obs=cv2.imread(str(SOURCE/'whole_pages'/page/'observed.png'),cv2.IMREAD_GRAYSCALE)
    if obs is None:
        raise RuntimeError(f'{page}: observed page unreadable')
    required=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
    with np.load(CACHE[page]) as z:
        missing=[k for k in required if k not in z.files]
        if missing:
            raise RuntimeError(f'{page}: cache missing {missing}')
        a={k:z[k].copy() for k in required}
    if any(v.shape!=obs.shape for v in a.values()):
        raise RuntimeError(f'{page}: contextual array shape mismatch')
    mask=a['mask']>0
    paper=background_surface(obs,mask)
    norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    runtime={
        'obs':obs,
        'norm':norm,
        'mask':mask,
        'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),
        'horizontal':a['g80_horizontal']>0,
        'vertical':a['g80_vertical']>0,
        'compact':a['g80_compact']>0,
        'preserve':a['g80_preserve']>0,
        'remove':a['g80_remove']>0,
        'ambiguous':a['g80_ambiguous']>0,
        'edge':cv2.Canny(obs,70,170)>0,
    }
    runtime['padded']={}
    for key in CHANNELS:
        arr=runtime[key].astype(np.float32)
        if key in ['obs','norm']:
            arr/=255.0
        runtime['padded'][key]=np.pad(arr,HALF,mode='reflect')
    PAGE_RUNTIME_CACHE[page]=runtime
    # Keep memory bounded while retaining the current working set.
    if len(PAGE_RUNTIME_CACHE)>4:
        oldest=next(iter(PAGE_RUNTIME_CACHE))
        if oldest!=page:
            PAGE_RUNTIME_CACHE.pop(oldest,None)
    return runtime

def exact_tensor(page,y,x):
    z=load_page_runtime(page)
    h,w=z['obs'].shape
    if not (0<=x<w and 0<=y<h):
        raise ValueError(f'Coordinate outside page: {(x,y)} for {(w,h)}')
    tensor=np.stack([z['padded'][k][y:y+PATCH,x:x+PATCH] for k in CHANNELS]).astype(np.float32)
    if tensor.shape!=(len(CHANNELS),PATCH,PATCH):
        raise RuntimeError(f'Unexpected tensor shape: {tensor.shape}')
    return tensor

def observed_patch_display(tensor,scale=12):
    patch=np.clip(tensor[0]*255.0,0,255).astype(np.uint8)
    out=cv2.resize(patch,(PATCH*scale,PATCH*scale),interpolation=cv2.INTER_NEAREST)
    c=HALF*scale+scale//2
    cv2.line(out,(c-2*scale,c),(c+2*scale,c),128,1)
    cv2.line(out,(c,c-2*scale),(c,c+2*scale),128,1)
    return cv2.cvtColor(out,cv2.COLOR_GRAY2RGB)

def normalise_channel_for_display(arr,key):
    a=np.asarray(arr,dtype=np.float32)
    if key in ['obs','norm']:
        z=np.clip(a,0,1)
    elif key=='depth':
        vmax=float(np.max(a))
        z=a/vmax if vmax>0 else np.zeros_like(a)
    else:
        z=(a>0).astype(np.float32)
    return (np.clip(z,0,1)*255).astype(np.uint8)

def tensor_contact_sheet(tensor,tile=180):
    cols=4
    rows=math.ceil(len(CHANNELS)/cols)
    header=28
    sheet=np.full((rows*(tile+header),cols*tile,3),255,np.uint8)
    for i,key in enumerate(CHANNELS):
        r,c=divmod(i,cols)
        img=normalise_channel_for_display(tensor[i],key)
        img=cv2.resize(img,(tile,tile),interpolation=cv2.INTER_NEAREST)
        img=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        cy=HALF*tile//PATCH+tile//(2*PATCH)
        cv2.drawMarker(img,(cy,cy),(255,0,255),cv2.MARKER_CROSS,max(8,tile//12),1)
        y0=r*(tile+header)
        x0=c*tile
        sheet[y0:y0+tile,x0:x0+tile]=img
        cv2.putText(sheet,key,(x0+5,y0+tile+20),cv2.FONT_HERSHEY_SIMPLEX,.55,(20,20,20),1,cv2.LINE_AA)
    return sheet

## 3. Six-band live rendering, zoom and pan

In [ ]:
def ordered_breakpoints(*values):
    vals=np.clip(np.asarray(values,dtype=float)/100.0,0.0,1.0)
    vals=np.sort(vals)
    # Prevent zero-width bands without changing the visible slider values materially.
    eps=1e-6
    for i in range(1,len(vals)):
        if vals[i]<=vals[i-1]: vals[i]=min(1.0,vals[i-1]+eps)
    return vals

def band_index(prob,bounds):
    return int(np.searchsorted(bounds,float(prob),side='right'))

def band_labels(bounds):
    edges=np.r_[0.0,bounds,1.0]
    return [f'{STATE_NAMES[i]} [{edges[i]:.3f}–{edges[i+1]:.3f}{"]" if i==5 else ")"}' for i in range(6)]

def dilate_mask(mask,radius):
    radius=int(radius)
    if radius<=0:return mask
    k=np.ones((2*radius+1,2*radius+1),np.uint8)
    return cv2.dilate(mask.astype(np.uint8),k)>0

def page_prob_arrays(page):
    z=load_page_runtime(page)
    h,w=z['obs'].shape
    g=PAGE_ROWS[page]
    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    prob=np.full((h,w),np.nan,np.float32)
    prob[ys,xs]=g.p_print.to_numpy(np.float32)
    valid=np.zeros((h,w),bool); valid[ys,xs]=True
    return z['obs'],prob,valid,g

def make_full_views(page,bounds,alpha,radius):
    observed,prob,valid,g=page_prob_arrays(page)
    base=np.repeat(observed[:,:,None],3,axis=2)
    state=np.full(observed.shape,-1,np.int8)
    state[valid]=np.searchsorted(bounds,prob[valid],side='right').astype(np.int8)

    colour=base.copy(); active=np.zeros(observed.shape,bool)
    for i,rgb in enumerate(STATE_RGB):
        m=dilate_mask(state==i,radius)
        colour[m]=rgb; active|=m
    six=base.copy()
    mixed=cv2.addWeighted(base,1-float(alpha),colour,float(alpha),0)
    six[active]=mixed[active]

    opinion=base.copy()
    value_rgb=[(255,255,255),(255,255,255),(210,210,210),(105,105,105),(0,0,0),(0,0,0)]
    for i,rgb in enumerate(value_rgb):
        m=dilate_mask(state==i,radius)
        opinion[m]=rgb

    extreme=base.copy()
    extreme_colours=[(255,255,255),STATE_RGB[1],STATE_RGB[2],STATE_RGB[3],STATE_RGB[4],(0,0,0)]
    for i,rgb in enumerate(extreme_colours):
        m=dilate_mask(state==i,radius)
        extreme[m]=rgb

    return opinion,extreme,six,state

def viewport_box(shape,zoom,pan_x,pan_y):
    h,w=shape[:2]
    zoom=max(1.0,float(zoom))
    cw=max(1,int(round(w/zoom))); ch=max(1,int(round(h/zoom)))
    max_x=max(0,w-cw); max_y=max(0,h-ch)
    x0=int(round(max_x*float(pan_x)/100.0)); y0=int(round(max_y*float(pan_y)/100.0))
    return x0,y0,min(w,x0+cw),min(h,y0+ch)

def crop_and_resize(img,box):
    x0,y0,x1,y1=box
    crop=img[y0:y1,x0:x1]
    h,w=crop.shape[:2]
    scale=min(VIEWPORT_MAX_WIDTH/max(w,1),VIEWPORT_MAX_HEIGHT/max(h,1),1.0 if max(h,w)<900 else 10.0)
    # At zoom=1 large pages are downsampled; zoomed crops are enlarged only up to the viewport.
    if w>VIEWPORT_MAX_WIDTH or h>VIEWPORT_MAX_HEIGHT:
        scale=min(VIEWPORT_MAX_WIDTH/w,VIEWPORT_MAX_HEIGHT/h)
    elif w<VIEWPORT_MAX_WIDTH and h<VIEWPORT_MAX_HEIGHT:
        scale=min(VIEWPORT_MAX_WIDTH/w,VIEWPORT_MAX_HEIGHT/h)
    rw=max(1,int(round(w*scale))); rh=max(1,int(round(h*scale)))
    return cv2.resize(crop,(rw,rh),interpolation=cv2.INTER_NEAREST), {'x0':x0,'y0':y0,'x1':x1,'y1':y1,'render_w':rw,'render_h':rh}

def add_legend(img,page,bounds):
    labels=band_labels(bounds)
    h,w=img.shape[:2]
    banner=95
    canvas=np.full((h+banner,w,3),255,np.uint8)
    canvas[banner:]=img
    cv2.putText(canvas,f'{page} — six-state confidence overlay',(10,28),cv2.FONT_HERSHEY_SIMPLEX,.65,(25,25,25),1,cv2.LINE_AA)
    x=10; y=62
    for label,rgb in zip(labels,STATE_RGB):
        item_w=max(170,min(290,45+len(label)*7))
        if x+item_w>w:
            y+=25; x=10
        cv2.rectangle(canvas,(x,y-14),(x+18,y+4),rgb,-1)
        cv2.putText(canvas,label,(x+24,y+3),cv2.FONT_HERSHEY_SIMPLEX,.40,(25,25,25),1,cv2.LINE_AA)
        x+=item_w
    return canvas

def render(page,b1,b2,b3,b4,b5,alpha,radius,zoom,pan_x,pan_y):
    bounds=ordered_breakpoints(b1,b2,b3,b4,b5)
    opinion,extreme,six,state=make_full_views(page,bounds,alpha,radius)
    box=viewport_box(opinion.shape,zoom,pan_x,pan_y)
    opinion_v,meta=crop_and_resize(opinion,box)
    extreme_v,_=crop_and_resize(extreme,box)
    six_v,_=crop_and_resize(six,box)
    six_v=add_legend(six_v,page,bounds)
    labels='  |  '.join(band_labels(bounds))
    status=f'**Page:** `{page}`  |  **Viewport:** x={box[0]}:{box[2]}, y={box[1]}:{box[3]}  |  {labels}'
    return opinion_v,extreme_v,six_v,status,meta

## 4. Click inspector

In [ ]:
def display_click_to_page(evt_index,meta,legend_offset=0):
    if evt_index is None:
        raise ValueError('No click coordinates received.')
    x_disp,y_disp=[float(v) for v in evt_index[:2]]
    y_disp=max(0.0,y_disp-float(legend_offset))
    rw=max(1,float(meta['render_w'])); rh=max(1,float(meta['render_h']))
    x0,y0,x1,y1=[float(meta[k]) for k in ['x0','y0','x1','y1']]
    x=int(np.clip(round(x0+x_disp/rw*(x1-x0)),x0,x1-1))
    y=int(np.clip(round(y0+y_disp/rh*(y1-y0)),y0,y1-1))
    return x,y,(int(round(x_disp)),int(round(y_disp)))

def nearest_candidate(page,x,y):
    lookup=PAGE_LOOKUP[page]
    exact=lookup.get((x,y))
    if exact is not None:
        return exact,x,y,0.0,False
    if not SNAP_TO_NEAREST_ELIGIBLE:
        return None,x,y,None,False
    g=PAGE_ROWS[page]
    dx=g.x.to_numpy(float)-x; dy=g.y.to_numpy(float)-y
    dist2=dx*dx+dy*dy
    j=int(np.argmin(dist2)); d=float(np.sqrt(dist2[j]))
    if d<=MAX_SNAP_DISTANCE:
        row=g.iloc[j]
        return j,int(row.x),int(row.y),d,True
    return None,x,y,d,False

def inspector_payload(page,x_clicked,y_clicked,bounds):
    g=PAGE_ROWS[page]
    idx,x,y,d,snapped=nearest_candidate(page,x_clicked,y_clicked)
    if idx is None:
        result={
            'page':page,
            'clicked_x':int(x_clicked),'clicked_y':int(y_clicked),
            'eligible_candidate':False,
            'nearest_eligible_distance':None if d is None else round(d,3),
            'message':'The clicked page pixel is not an eligible V8.2 candidate. No p_print or truth label exists for this exact coordinate.'
        }
        return result,None,None,None
    row=g.iloc[int(idx)]
    tensor=exact_tensor(page,int(y),int(x))
    bidx=band_index(float(row.p_print),bounds)
    truth_name=TRUTH_NAMES.get(int(row.truth),f'UNKNOWN_{row.truth}')
    result={
        'page':page,
        'clicked_x':int(x_clicked),'clicked_y':int(y_clicked),
        'candidate_x':int(x),'candidate_y':int(y),
        'snapped_to_nearest_eligible':bool(snapped),
        'snap_distance_pixels':round(float(d),3),
        'eligible_candidate':True,
        'p_print':float(row.p_print),
        'p_annotation':float(1.0-row.p_print),
        'raw_print':float(row.raw_print),
        'truth_numeric':int(row.truth),
        'truth_label':truth_name,
        'probability_band_index':int(bidx)+1,
        'probability_band':band_labels(bounds)[bidx],
        'fold':int(row.fold),
        'source_group':str(row.source_group),
        'saved_decision':str(row.decision) if 'decision' in row.index else 'NOT_PRESENT',
        'patch_shape':list(tensor.shape),
        'patch_centre_index':[HALF,HALF],
    }
    rows=[]
    for i,key in enumerate(CHANNELS):
        a=tensor[i]
        rows.append({
            'channel':key,'centre':float(a[HALF,HALF]),
            'min':float(np.min(a)),'max':float(np.max(a)),'mean':float(np.mean(a))
        })
    result['channel_statistics']=rows
    return result,tensor,observed_patch_display(tensor),tensor_contact_sheet(tensor)

def inspect_click(page,b1,b2,b3,b4,b5,meta,source_view,evt:gr.SelectData):
    bounds=ordered_breakpoints(b1,b2,b3,b4,b5)
    legend_offset=95 if source_view=='six' else 0
    x,y,display_xy=display_click_to_page(evt.index,meta,legend_offset=legend_offset)
    result,tensor,patch,sheet=inspector_payload(page,x,y,bounds)
    result['clicked_display_xy']=list(display_xy)
    result['clicked_view']=source_view
    if not result['eligible_candidate']:
        msg=(f"### Pixel inspector\n- **Page:** `{page}`\n- **Clicked full-page coordinate:** `({x}, {y})`\n"
             f"- **Eligible V8.2 candidate:** No\n- **Nearest eligible distance:** `{result['nearest_eligible_distance']}` pixels\n\n"
             "No probability or truth label is assigned to a non-eligible pixel.")
        return msg,None,None,result,result
    snap=(f" → resolved candidate `({result['candidate_x']}, {result['candidate_y']})` "
          f"at distance {result['snap_distance_pixels']:.2f}px" if result['snapped_to_nearest_eligible'] else '')
    msg=(
        f"### Pixel inspector\n"
        f"- **Clicked full-page `(x, y)`:** `({x}, {y})`{snap}\n"
        f"- **Calibrated `p_print`:** `{result['p_print']:.12g}`\n"
        f"- **Calibrated `p_annotation`:** `{result['p_annotation']:.12g}`\n"
        f"- **Truth label:** **{result['truth_label']}** (`{result['truth_numeric']}`)\n"
        f"- **Current probability band:** **{result['probability_band']}**\n"
        f"- **Raw PRINT score:** `{result['raw_print']:.12g}`\n"
        f"- **Fold / source group:** `{result['fold']}` / `{result['source_group']}`\n"
        f"- **Saved V8.2 decision:** `{result['saved_decision']}`\n"
        f"- **Tensor:** `{result['patch_shape']}` in channel order `{CHANNELS}`\n\n"
        "Truth is post-hoc development evidence only; it did not enter the model input."
    )
    return msg,patch,sheet,result,result

def inspect_from_opinion(page,b1,b2,b3,b4,b5,meta,evt:gr.SelectData):
    return inspect_click(page,b1,b2,b3,b4,b5,meta,'opinion',evt)
def inspect_from_extreme(page,b1,b2,b3,b4,b5,meta,evt:gr.SelectData):
    return inspect_click(page,b1,b2,b3,b4,b5,meta,'extreme',evt)
def inspect_from_six(page,b1,b2,b3,b4,b5,meta,evt:gr.SelectData):
    return inspect_click(page,b1,b2,b3,b4,b5,meta,'six',evt)

## 5. Snapshot and inspected-pixel preservation

In [ ]:
def save_snapshot(page,b1,b2,b3,b4,b5,alpha,radius,current_inspection):
    bounds=ordered_breakpoints(b1,b2,b3,b4,b5)
    opinion,extreme,six,_=make_full_views(page,bounds,alpha,radius)
    out=SESSION_DIR/page
    out.mkdir(parents=True,exist_ok=True)
    cv2.imwrite(str(out/'classifier_opinion_on_page.png'),cv2.cvtColor(opinion,cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(out/'extreme_state_review_preview.png'),cv2.cvtColor(extreme,cv2.COLOR_RGB2BGR))
    six_legend=add_legend(six,page,bounds)
    cv2.imwrite(str(out/'six_state_confidence_overlay.png'),cv2.cvtColor(six_legend,cv2.COLOR_RGB2BGR))
    record={
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'source_v82_run':str(SOURCE),'page':page,
        'breakpoints':bounds.tolist(),'alpha':float(alpha),'radius':int(radius),
        'diagnostic_only':True,'current_inspection':current_inspection,
    }
    (out/'band_experiment.json').write_text(json.dumps(record,indent=2))
    if current_inspection and current_inspection.get('eligible_candidate'):
        x=int(current_inspection['candidate_x']); y=int(current_inspection['candidate_y'])
        tensor=exact_tensor(page,y,x)
        np.save(out/f'pixel_y{y}_x{x}_tensor_11x33x33.npy',tensor)
        cv2.imwrite(str(out/f'pixel_y{y}_x{x}_observed_patch.png'),cv2.cvtColor(observed_patch_display(tensor),cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(out/f'pixel_y{y}_x{x}_tensor_contact_sheet.png'),cv2.cvtColor(tensor_contact_sheet(tensor),cv2.COLOR_RGB2BGR))
        (out/f'pixel_y{y}_x{x}_inspection.json').write_text(json.dumps(current_inspection,indent=2))
    return f'Saved snapshot to `{out}`'

def step_page(current,delta):
    i=PAGES.index(current) if current in PAGES else 0
    return PAGES[(i+int(delta))%len(PAGES)]

## 6. Launch Gradio explorer

In [ ]:
with gr.Blocks(title='V8.2b Six-Band Confidence Explorer + Click Inspector') as demo:
    gr.Markdown('# V8.2b Interactive Six-Band Confidence Explorer')
    gr.Markdown(
        'Development diagnostic only. Click any image to inspect the exact OOF candidate. '
        'If a dilated display mark is clicked, the inspector reports both the clicked coordinate and any nearby eligible source coordinate.'
    )
    viewport_state=gr.State({})
    inspection_state=gr.State({})

    with gr.Row():
        prev_btn=gr.Button('◀ Previous')
        page=gr.Dropdown(PAGES,value=PAGES[0],label='Annotated page',filterable=True)
        next_btn=gr.Button('Next ▶')
        refresh_btn=gr.Button('↻ Refresh',variant='secondary')

    with gr.Accordion('Six live confidence boundaries',open=True):
        gr.Markdown('The five values are sorted before rendering, creating six ordered bands.')
        with gr.Row():
            b1=gr.Slider(0,100,DEFAULT_BREAKPOINTS_PERCENT[0],step=1,label='Boundary 1 (%)')
            b2=gr.Slider(0,100,DEFAULT_BREAKPOINTS_PERCENT[1],step=1,label='Boundary 2 (%)')
            b3=gr.Slider(0,100,DEFAULT_BREAKPOINTS_PERCENT[2],step=1,label='Boundary 3 (%)')
            b4=gr.Slider(0,100,DEFAULT_BREAKPOINTS_PERCENT[3],step=1,label='Boundary 4 (%)')
            b5=gr.Slider(0,100,DEFAULT_BREAKPOINTS_PERCENT[4],step=1,label='Boundary 5 (%)')

    with gr.Row():
        alpha=gr.Slider(.10,1.00,DEFAULT_ALPHA,step=.02,label='Colour overlay alpha')
        radius=gr.Slider(0,3,DEFAULT_RADIUS,step=1,label='Display visibility radius (pixels)')
        zoom=gr.Slider(1,8,DEFAULT_ZOOM,step=.25,label='Zoom')
        pan_x=gr.Slider(0,100,DEFAULT_PAN_X,step=1,label='Pan left ↔ right (%)')
        pan_y=gr.Slider(0,100,DEFAULT_PAN_Y,step=1,label='Pan top ↕ bottom (%)')

    status=gr.Markdown()
    with gr.Row():
        opinion_img=gr.Image(label='classifier_opinion_on_page.png — click to inspect',type='numpy')
        extreme_img=gr.Image(label='extreme_state_review_preview.png — click to inspect',type='numpy')
        six_img=gr.Image(label='six_state_confidence_overlay.png — click to inspect',type='numpy')

    gr.Markdown('## Click inspector')
    inspector_md=gr.Markdown('Click an image above.')
    with gr.Row():
        patch_img=gr.Image(label='Exact 33×33 observed patch (enlarged; cross marks centre)',type='numpy')
        tensor_img=gr.Image(label='Exact 11-channel 33×33 model-input tensor',type='numpy')
    inspector_json=gr.JSON(label='Exact pixel record and channel statistics')

    with gr.Row():
        save_btn=gr.Button('💾 Save full-resolution snapshot + current inspection',variant='primary')
        save_status=gr.Markdown()

    render_inputs=[page,b1,b2,b3,b4,b5,alpha,radius,zoom,pan_x,pan_y]
    render_outputs=[opinion_img,extreme_img,six_img,status,viewport_state]
    inspect_inputs=[page,b1,b2,b3,b4,b5,viewport_state]
    inspect_outputs=[inspector_md,patch_img,tensor_img,inspector_json,inspection_state]

    for control in [page,b1,b2,b3,b4,b5,alpha,radius,zoom,pan_x,pan_y]:
        control.change(render,inputs=render_inputs,outputs=render_outputs,show_progress='hidden')
    refresh_btn.click(render,inputs=render_inputs,outputs=render_outputs)

    prev_btn.click(lambda p:step_page(p,-1),inputs=page,outputs=page).then(render,inputs=render_inputs,outputs=render_outputs)
    next_btn.click(lambda p:step_page(p,1),inputs=page,outputs=page).then(render,inputs=render_inputs,outputs=render_outputs)

    opinion_img.select(inspect_from_opinion,inputs=inspect_inputs,outputs=inspect_outputs)
    extreme_img.select(inspect_from_extreme,inputs=inspect_inputs,outputs=inspect_outputs)
    six_img.select(inspect_from_six,inputs=inspect_inputs,outputs=inspect_outputs)

    save_btn.click(save_snapshot,inputs=[page,b1,b2,b3,b4,b5,alpha,radius,inspection_state],outputs=save_status)
    demo.load(render,inputs=render_inputs,outputs=render_outputs)

demo.queue(default_concurrency_limit=2).launch(share=True,debug=True)